<a href="https://colab.research.google.com/github/Tuanyatuanya/MSSP607/blob/main/Participation_Week_09_Activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/tuanyatuanya/MSSP607.git

Mounted at /content/drive
Cloning into 'MSSP607'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 152 (delta 76), reused 31 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 1.30 MiB | 6.85 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [1]:
import webbrowser
import os
import requests
import bs4
from pathlib import Path
from urllib.parse import quote_plus

In [3]:
def download_imgur_search_images(query):
    # Set up download folder
    folder_name = query.replace(" ", "_")
    download_path = Path(folder_name)
    download_path.mkdir(exist_ok=True)

    # Build Imgur search URL
    base_url = "https://imgur.com/search?q="
    search_url = base_url + quote_plus(query)

    print(f"Searching Imgur for: {query}")
    print(f"URL: {search_url}")

    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; ImageDownloader/1.0)"
    }

    # Download the search page HTML
    res = requests.get(search_url, headers=headers)
    res.raise_for_status()

    soup = bs4.BeautifulSoup(res.text, "html.parser")

    # Find image tags – Imgur often stores them with "src" or "data-src"
    image_urls = set()  # use a set to avoid duplicates
    for img in soup.select("img"):
        src = img.get("data-src") or img.get("src")
        if not src:
            continue

        # We only want Imgur image URLs
        if "i.imgur.com" in src:
            # Some links start with //, add https:
            if src.startswith("//"):
                src = "https:" + src
            elif src.startswith("/"):
                src = "https://imgur.com" + src

            image_urls.add(src)

    print(f"Found {len(image_urls)} image URLs.")

    # Download each image
    for i, image_url in enumerate(image_urls, start=1):
        print(f"Downloading image {i}: {image_url}")
        try:
            img_res = requests.get(image_url, headers=headers)
            img_res.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"  Failed to download {image_url}: {e}")
            continue

        # Get file extension from URL (e.g., .jpg, .png)
        ext = os.path.splitext(image_url)[1]
        if not ext:
            ext = ".jpg"

        file_name = download_path / f"image_{i}{ext}"
        with open(file_name, "wb") as f:
            for chunk in img_res.iter_content(100000):
                f.write(chunk)

    print("Done!")

if __name__ == "__main__":
    search_term = input("Enter a search term for images (e.g. cats, sunset): ")
    download_imgur_search_images(search_term)

Enter a search term for images (e.g. cats, sunset): cats
Searching Imgur for: cats
URL: https://imgur.com/search?q=cats
Found 52 image URLs.
Done!
